In [2]:
# what is fst 
# fst -  finite state transducer 
# "any stem + छु" → "any stem + <SLOT>छु"
# it splits root and suffix 
# this works for  every verb ending in छु -->.गर्छु, खान्छु, जान्छु,
#  गर्छु --FST runs ---> गर्  +  <SLOT>  +  छु

In [3]:
# approach 
# extract suffix rules from paradigm.csv
# Build FST using those rules

In [4]:
import pandas as pd 

df = pd.read_csv('../data/paradigm.csv')
df

,stem,stem_nepali,slot,surface_form,gloss,verified
0,gar,गर्,1SG.PRES,गर्छु,I do,YES
1,gar,गर्,2SG.PRES,गर्छस्,you do (informal),YES
2,gar,गर्,2SG.PRES.MID,गर्छौ,you do (friendly),YES
3,gar,गर्,3SG.PRES,गर्छ,he/she does,YES
4,gar,गर्,1PL.PRES,गर्छौं,we do,YES
...,...,...,...,...,...,...
145,sahar,सहर,ACC,सहरलाई,city (object),YES
146,sahar,सहर,GEN,सहरको,of the city,YES
147,sahar,सहर,LOC,सहरमा,in/at the city,YES
148,sahar,सहर,ABL,सहरबाट,from the city,YES


In [5]:
df[['slot','surface_form','stem_nepali']]

,slot,surface_form,stem_nepali
0,1SG.PRES,गर्छु,गर्
1,2SG.PRES,गर्छस्,गर्
2,2SG.PRES.MID,गर्छौ,गर्
3,3SG.PRES,गर्छ,गर्
4,1PL.PRES,गर्छौं,गर्
...,...,...,...
145,ACC,सहरलाई,सहर
146,GEN,सहरको,सहर
147,LOC,सहरमा,सहर
148,ABL,सहरबाट,सहर


In [6]:
def extract_suffix(stem, surface_form):
    if surface_form.startswith(stem):
        return surface_form[len(stem):]
    else:
        return None

df['suffix'] = df.apply(
    lambda row: extract_suffix(row['stem_nepali'], row['surface_form']),axis=1)

df

,stem,stem_nepali,slot,surface_form,gloss,verified,suffix
0,gar,गर्,1SG.PRES,गर्छु,I do,YES,छु
1,gar,गर्,2SG.PRES,गर्छस्,you do (informal),YES,छस्
2,gar,गर्,2SG.PRES.MID,गर्छौ,you do (friendly),YES,छौ
3,gar,गर्,3SG.PRES,गर्छ,he/she does,YES,छ
4,gar,गर्,1PL.PRES,गर्छौं,we do,YES,छौं
...,...,...,...,...,...,...,...
145,sahar,सहर,ACC,सहरलाई,city (object),YES,लाई
146,sahar,सहर,GEN,सहरको,of the city,YES,को
147,sahar,सहर,LOC,सहरमा,in/at the city,YES,मा
148,sahar,सहर,ABL,सहरबाट,from the city,YES,बाट


In [7]:
df['suffix'].isna().sum()

10

In [8]:
print(df[['stem_nepali', 'slot', 'surface_form', 'suffix']].head(20))

   stem_nepali          slot surface_form   suffix
0          गर्      1SG.PRES        गर्छु       छु
1          गर्      2SG.PRES       गर्छस्      छस्
2          गर्  2SG.PRES.MID        गर्छौ       छौ
3          गर्      3SG.PRES         गर्छ        छ
4          गर्      1PL.PRES       गर्छौं      छौं
5          गर्      2PL.PRES        गर्छौ       छौ
6          गर्      3PL.PRES       गर्छन्      छन्
7          गर्     2HON.PRES   गर्नुहुन्छ  नुहुन्छ
8          गर्     3HON.PRES   गर्नुहुन्छ  नुहुन्छ
9          गर्      1SG.PAST         गरें     None
10         गर्      2SG.PAST        गर्यौ       यौ
11         गर्      3SG.PAST        गर्यो       यो
12         गर्      1PL.PAST       गर्यौं      यौं
13         गर्      2PL.PAST        गर्यौ       यौ
14         गर्      3PL.PAST          गरे     None
15         गर्     2HON.PAST     गर्नुभयो    नुभयो
16         गर्     3HON.PAST     गर्नुभयो    नुभयो
17         गर्       1SG.FUT      गर्नेछु     नेछु
18         गर्       2SG.FUT   

In [9]:
# next problem 
# when the tokenzier encountered गर्नुभयो , there are 2 rules that applies , 
# Rule 1: यो  → HON.PAST
# Rule 2: नुभयो → HON.PAST

# गर्नुभ + <SLOT> + यो -- this is wrong
# गर् + <SLOT> + नुभयो -- this is right , but how tokenizzer knows 

In [26]:
suffix_rules = df[['suffix','slot']].drop_duplicates()
suffix_rules = suffix_rules.sort_values('suffix',key = lambda x:x.str.len(), ascending=False)
suffix_rules = suffix_rules[suffix_rules['suffix'].str.len() > 0]
suffix_rules = suffix_rules.reset_index(drop=True)


In [27]:


print(f"Total unique suffix rules: {len(suffix_rules)}")
print(suffix_rules)

Total unique suffix rules: 54
      suffix          slot
0   उनुहुनेछ      3HON.FUT
1   उनुहुन्छ     3HON.PRES
2   उनुहुन्छ     2HON.PRES
3   उनुहुनेछ      2HON.FUT
4    नुहुनेछ      3HON.FUT
5    नुहुनेछ      2HON.FUT
6    नुहुन्छ     2HON.PRES
7    नुहुन्छ     3HON.PRES
8     उनेछस्       2SG.FUT
9     उनेछौं       1PL.FUT
10    उनेछन्       3PL.FUT
11    उनुभयो     3HON.PAST
12    उनुभयो     2HON.PAST
13     नेछन्       3PL.FUT
14     उँछन्      3PL.PRES
15     न्छौं      1PL.PRES
16     उनेछु       1SG.FUT
17     न्छस्      2SG.PRES
18     उँछस्      2SG.PRES
19     नुभयो     3HON.PAST
20     नेछौं       1PL.FUT
21     नेछस्       2SG.FUT
22     उँछौं      1PL.PRES
23     न्छन्      3PL.PRES
24     नुभयो     2HON.PAST
25      उँछौ  2SG.PRES.MID
26      उँछु      1SG.PRES
27      उँछौ      2PL.PRES
28      न्छौ      2PL.PRES
29      न्छौ  2SG.PRES.MID
30      उनेछ       3SG.FUT
31      नेछु       1SG.FUT
32      न्छु      1SG.PRES
33       छस्      2SG.PRES
34       न्छ      3SG.PRE

In [28]:
# ## discovery: allomorphy in Nepali 3PL.PAST
# - vowel stems use: ए (e.g. गए, आए)
# - consonant stems use: े (e.g. गरे, बोले)
# - same grammatical slot, different surface form
# - FST must handle both rules

In [32]:
# main fst algo
stopwords = {
    'अहिले', 'तेसैले', 'किनभने', 'त्यसैले', 'यसैले',
    'तेतिबेला', 'कहिले', 'पहिले', 'त्यो', 'यो', 'यी',
    'ती', 'जो', 'जति', 'सबै', 'कति', 'भोलि', 'हिजो',
    'आज', 'पहिलो', 'दोस्रो', 'तेस्रो', 'चाँडै', 'ढिलो',
    'तुरुन्त', 'फेरि', 'अझै', 'झन्', 'नि', 'पनि',
    'मलाई', 'मले',
    'उहाँले', 'उहाँलाई', 'उहाँको', 'उहाँ',
    'उसले', 'उसलाई', 'उसको',
    'तिमीले', 'तिमीलाई',
    'हामीले', 'हामीलाई',
    'उनले', 'उनलाई', 'उनको',
}

def fst(text, suffix_rules):
    words = text.split()
    result = []
    for word in words:
        if word in stopwords:
            result.append(word)
            continue
        marked = False
        for _, row in suffix_rules.iterrows():
            suffix  = row['suffix']
            if isinstance(suffix, str) and  word.endswith(suffix):
                # find where suffix starts
                root = word[:-len(suffix)]
                if len(root) > 0: # make sure root is not empty 
                    marked_word = root + '<SLOT>' + suffix
                    result.append(marked_word)
                    marked = True
                    break
        if not marked:
            result.append(word) # no rule matched, keep as is 
    return ' '.join(result)

    


In [36]:
test_sentences = [
    "म खाना खान्छु",
    "उहाँले गर्नुभयो",
    "हामी घर जान्छौं",
    "मलाई चिया भन्दा कफी बढी मन पर्छ",
    "त्यो कुकुर धेरै मिलनसार छ",
    "काठमाडौंको ट्राफिकले हैरान बनाउँछ",
]

for sentence in test_sentences:
    output = fst(sentence, suffix_rules)
    print(f"Input:  {sentence}")
    print(f"Output: {output}")
    print()

# key notes
# done spliting root + suffx of verbs
# skipped splittingof pronounce 
# and leaved noun as same, a lot of nouns to add


Input:  म खाना खान्छु
Output: म खाना खा<SLOT>न्छु

Input:  उहाँले गर्नुभयो
Output: उहाँले गर्<SLOT>नुभयो

Input:  हामी घर जान्छौं
Output: हामी घर जा<SLOT>न्छौं

Input:  मलाई चिया भन्दा कफी बढी मन पर्छ
Output: मलाई चिया भन्दा कफी बढी मन पर्<SLOT>छ

Input:  त्यो कुकुर धेरै मिलनसार छ
Output: त्यो कुकुर धेरै मिलनसार छ

Input:  काठमाडौंको ट्राफिकले हैरान बनाउँछ
Output: काठमाडौं<SLOT>को ट्राफिक<SLOT>ले हैरान बना<SLOT>उँछ

